In [2]:
from langchain_neo4j import Neo4jGraph
from configuration.config import *

graph = Neo4jGraph(
    url=NEO4J_CONFIG["uri"],
    username=NEO4J_CONFIG["auth"][0],
    password=NEO4J_CONFIG["auth"][1],
)

In [3]:
print(graph.schema)

Node properties:
Person {name: STRING, birth: STRING}
Movie {title: STRING, genre: LIST, rating: FLOAT, year: INTEGER}
User {name: STRING, userId: STRING, email: STRING}
Category1 {name: STRING, id: INTEGER}
Category2 {name: STRING, id: INTEGER}
Tag {name: STRING, id: STRING}
Category3 {name: STRING, id: INTEGER}
BaseAttrName {name: STRING, id: INTEGER}
BaseAttrValue {name: STRING, id: INTEGER}
SPU {name: STRING, id: INTEGER}
SKU {name: STRING, id: INTEGER}
Trademark {name: STRING, id: INTEGER}
SaleAttrName {name: STRING, id: INTEGER}
SaleAttrValue {name: STRING, id: INTEGER}
Relationship properties:
ACTED_IN {role: STRING, award: BOOLEAN}
DIRECTED {award: BOOLEAN}
The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:FOLLOWS]->(:Person)
(:Category1)-[:Have]->(:BaseAttrName)
(:Category2)-[:Belong]->(:Category1)
(:Category2)-[:Have]->(:BaseAttrName)
(:Category3)-[:Belong]->(:Category2)
(:Category3)-[:Have]->(:BaseAttrName)
(:BaseAttrName)-[:Have]

In [4]:
# 查询
cypher = "MATCH (n) RETURN n LIMIT 5"
graph.query(cypher)

[{'n': {'name': '张艺谋', 'birth': '1951-11-14'}},
 {'n': {'name': '陈凯歌', 'birth': '1952-08-12'}},
 {'n': {'name': '巩俐', 'birth': '1965-12-31'}},
 {'n': {'name': '葛优', 'birth': '1957-04-19'}},
 {'n': {'name': '章子怡', 'birth': '1979-02-09'}}]

In [5]:
from langchain_deepseek import ChatDeepSeek

# 定义大模型
llm = ChatDeepSeek(model='deepseek-chat', temperature=0,api_key=API_KEY)

In [6]:
from langchain_neo4j import GraphCypherQAChain

# 定义chain
chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)

In [9]:
result = chain.invoke({"query": "华为有哪些产品？"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Trademark {name: '华为'})<-[:Belong]-(spu:SPU)
RETURN spu.name AS product
Full Context:
[{'product': '华为智慧屏 4K全面屏智能电视机'}, {'product': 'HAWEIAI Book 2025英特尔14 Pro满血独显笔记本'}, {'product': '华为HUAWEI二手笔记本MateBook13触屏2K全面屏'}, {'product': '华为Mate 40 pro'}]

> Finished chain.
